# 02 — Cleaning decisions

Each section applies **one step of the pipeline** and shows its effect: before → after, plus rows affected.

**This notebook contains no cleaning logic.** It imports the functions from [`clean_pipeline.py`](clean_pipeline.py), so it always shows exactly what the pipeline does. Change a rule in the script, re-run this notebook, and the evidence updates.

Evidence for *why* each rule exists: `01_profiling.ipynb` · reasoning: [`cleaning_report.md`](cleaning_report.md) · rule list: [`cleaning_rules.md`](cleaning_rules.md).

All 16 rules (R01–R16) are implemented. Result: **180,519 rows × 53 columns → 180,519 rows × 42 columns.**

In [1]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "Clean" else Path.cwd()
sys.path.insert(0, str(ROOT / "Clean"))
import clean_pipeline as cp

raw = cp.load_raw()
print(f"raw: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

raw: 180,519 rows x 53 columns


## Step 0: Fix shifted address rows (R10)
Runs **first** because it needs `Customer Street`, which R01 drops.

In [2]:
addr = ["Customer Street", "Customer City", "Customer State", "Customer Zipcode"]
shifted = raw["Customer State"].str.fullmatch(r"\d{5}")
s0 = cp.fix_shifted_addresses(raw.copy())
print(f"{shifted.sum()} rows fixed")
pd.concat({"before": raw.loc[shifted, addr], "after": s0.loc[shifted, addr]}, axis=1)

3 rows fixed


before                                                         after                                              
      Customer Street Customer City Customer State Customer Zipcode Customer Street Customer City Customer State Customer Zipcode
35704       Elk Grove            CA          95758              NaN       Elk Grove     Elk Grove             CA          95758.0
46440       Elk Grove            CA          95758              NaN       Elk Grove     Elk Grove             CA          95758.0
82511        El Monte            CA          91732              NaN        El Monte      El Monte             CA          91732.0

In these rows the street field held a city name and everything slid one column right. After the fix, city, state and zip are in the right place, and the street is dropped by R01 next.

## Step 1: Remove columns (53 → 38)

### R01: Drop personal data

In [3]:
s1 = cp.drop_pii(s0.copy())
print(f"{s0.shape[1]} → {s1.shape[1]} columns")
# show only what kind of data each column holds, never the values themselves
pd.DataFrame({
    "dropped column": cp.PII_COLUMNS,
    "distinct values": [raw[c].nunique() for c in cp.PII_COLUMNS],
})

53 → 48 columns


,dropped column,distinct values
0,Customer Email,1
1,Customer Password,1
2,Customer Fname,782
3,Customer Lname,1109
4,Customer Street,7458


Email and password are masked (1 distinct value); names and street identify people. None of the six questions needs them, and the repo is public.

### R02 + R03: Drop empty, constant and duplicate columns

In [4]:
s2 = cp.drop_unused_columns(s1.copy())  # asserts each duplicate is 100% identical before dropping
print(f"{s1.shape[1]} → {s2.shape[1]} columns")
pd.DataFrame(
    [(c, "no value") for c in cp.NO_VALUE_COLUMNS]
    + [(c, f"identical to '{k}'") for c, k in cp.DUPLICATE_COLUMNS.items()],
    columns=["dropped column", "reason"],
)

48 → 38 columns


,dropped column,reason
0,Product Description,no value
1,Product Status,no value
2,Product Image,no value
3,Order Zipcode,no value
4,Benefit per order,identical to 'Order Profit Per Order'
5,Sales per customer,identical to 'Order Item Total'
6,Order Customer Id,identical to 'Customer Id'
7,Product Category Id,identical to 'Category Id'
8,Order Item Product Price,identical to 'Product Price'
9,Order Item Cardprod Id,identical to 'Product Card Id'


### R04: Clear snake_case names

In [5]:
s3 = cp.rename_columns(s2.copy())
renames = pd.DataFrame(cp.RENAME_MAP.items(), columns=["raw name", "clean name"])
misleading = ["Order Profit Per Order", "Sales", "Order Item Total", "Type", "Late_delivery_risk", "Latitude", "Longitude"]
renames[renames["raw name"].isin(misleading)]

,raw name,clean name
9,Type,payment_type
15,Late_delivery_risk,is_late
21,Sales,sales
24,Order Item Total,net_sales
25,Order Profit Per Order,profit
36,Latitude,store_latitude
37,Longitude,store_longitude


The full mapping is `cp.RENAME_MAP`. The rows above are the names that were **misleading**, not just messy. `is_late` is also converted from 0/1 to true/false.

## Step 2: Fix types and formats (R05–R07)

In [6]:
s4 = cp.fix_formats(s3.copy())
cols = ["order_date", "shipping_date", "customer_zipcode", "net_sales", "discount_rate"]
pd.DataFrame({
    "type before": s3[cols].dtypes.astype(str),
    "type after": s4[cols].dtypes.astype(str),
    "example before": s3[cols].iloc[0].astype(str),
    "example after": s4[cols].iloc[0].astype(str),
})

,type before,type after,example before,example after
order_date,str,datetime64[us],1/31/2018 22:56,2018-01-31 22:56:00
shipping_date,str,datetime64[us],2/3/2018 22:56,2018-02-03 22:56:00
customer_zipcode,float64,string,725.0,00725
net_sales,float64,float64,314.6400146,314.64
discount_rate,float64,float64,0.039999999,0.04


### R06: Zip codes get their leading zeros back

In [7]:
short = s3["customer_zipcode"].astype(int).astype(str).str.len() < 5
print(f"{short.sum():,} zips restored")
pd.DataFrame({"before": s3.loc[short, "customer_zipcode"], "after": s4.loc[short, "customer_zipcode"],
              "country": s4.loc[short, "customer_country"]}).drop_duplicates("after").head(6)

75,508 zips restored


,before,after,country
0,725.0,00725,Puerto Rico
14,1960.0,01960,EE. UU.
16,729.0,00729,Puerto Rico
48,957.0,00957,Puerto Rico
101,784.0,00784,Puerto Rico
112,795.0,00795,Puerto Rico


### R07: Round money to 2 decimals

In [8]:
money = ["sales", "discount_amount", "net_sales", "profit"]
noisy = {c: int(((s3[c] - s3[c].round(2)).abs() > 1e-9).sum()) for c in money}
pd.DataFrame({
    "rows with float noise": noisy,
    "total before": s3[money].sum().round(2),
    "total after": s4[money].sum().round(2),
    "difference ($)": (s4[money].sum() - s3[money].sum()).round(2),
})

,rows with float noise,total before,total after,difference ($)
sales,155013,36784735.01,36784734.31,-0.70
discount_amount,66525,3730378.40,3730378.40,-0.00
net_sales,156939,33054402.38,33054402.04,-0.34
profit,163116,3966902.97,3966902.97,-0.00


The raw file stored money as low-precision floats (e.g. `294.980011`). Rounding removes the noise and moves the totals by **less than $1 on $36.8M**. From here on the **clean totals are the official control totals** (`Data/Cleaned/control_totals.json`).

The 4,613 rows where `net_sales` ≠ `sales − discount_amount` by exactly 1 cent are **source rounding** (half-cent discounts), not noise, so they are kept and validated with a ±$0.01 tolerance.

## Step 3: Fix values (R08, R09, R11, R12)

### R08: Trim and collapse whitespace

In [9]:
s5 = cp.standardise_text(s4.copy())
text_cols = [c for c in s4.columns if pd.api.types.is_string_dtype(s4[c]) and c not in ["order_country", "customer_country"]]
changed = pd.Series({c: (s4[c] != s5[c]).sum() for c in text_cols})
print(f"{changed.sum():,} cells changed")
changed[changed > 0].sort_values(ascending=False).to_frame("cells changed")

21,604 cells changed


,cells changed
order_region,17925
product_name,1774
category_name,1543
department_name,362


In [10]:
examples = pd.concat(
    [pd.DataFrame({"column": c, "before": s4.loc[s4[c] != s5[c], c], "after": s5.loc[s4[c] != s5[c], c]}).drop_duplicates("before")
     for c in ["category_name", "order_region", "product_name"]]
)
examples.assign(before=examples["before"].map(repr), after=examples["after"].map(repr)).head(8)

,column,before,after
146,category_name,'Cameras ','Cameras'
316,category_name,'As Seen on TV!','As Seen on TV!'
886,category_name,'Baby ','Baby'
1360,category_name,'Books ','Books'
1590,category_name,'CDs ','CDs'
55,order_region,'West of USA ','West of USA'
58,order_region,'US Center ','US Center'
478,order_region,'South of USA ','South of USA'


Unique counts don't change (no two categories merged). The fix gives clean labels and safe joins (`'Cameras '` ≠ `'Cameras'` in SQL).

### R09: Country names → English

In [11]:
mapping = pd.read_csv(cp.COUNTRY_MAP_FILE)
print(f"mapping file: {len(mapping)} names, {(mapping['country_es'] != mapping['country_en']).sum()} translated")
pairs = pd.DataFrame({"before": s4["order_country"], "after": s5["order_country"]})
pairs.value_counts().head(8).to_frame("items")

mapping file: 166 names, 103 translated


,,items
before,after,
Estados Unidos,United States,24840
Francia,France,13222
México,Mexico,13172
Alemania,Germany,9564
Australia,Australia,8497
Brasil,Brazil,7987
Reino Unido,United Kingdom,7302
China,China,5758


In [12]:
pd.crosstab(s4["customer_country"], s5["customer_country"]).rename_axis(index="before", columns="after")

after,Puerto Rico,United States
before,,
EE. UU.,0,111146
Puerto Rico,69373,0


The pipeline **fails loudly** if a country is missing from `mappings/country_es_en.csv`, so a new source file can't sneak Spanish names through.

### R11 + R12: Longitude sign and duplicate category name

In [13]:
s6 = cp.fix_known_errors(s5.copy())
flipped = s5["store_longitude"] > 0
print(f"R11: {flipped.sum()} store longitudes negated")
pd.concat({"before": s5.loc[flipped, ["customer_city", "customer_state", "store_longitude"]],
           "after": s6.loc[flipped, ["store_longitude"]]}, axis=1).drop_duplicates().head(5)

R11: 105 store longitudes negated


before                                          after
      customer_city customer_state store_longitude store_longitude
2034   Douglasville             GA       84.742676      -84.742676
4138      Las Vegas             NV      115.049789     -115.049789
8864          Tampa             FL       82.461998      -82.461998
18673    Washington             DC       77.092094      -77.092094
42977     Las Vegas             NV      115.263077     -115.263077

In [14]:
elec = s5["category_name"] == "Electronics"
pd.DataFrame({"category_id": s5.loc[elec, "category_id"], "before": s5.loc[elec, "category_name"],
              "after": s6.loc[elec, "category_name"]}).value_counts().to_frame("items")

,,,items
category_id,before,after,
37,Electronics,Electronics (Outdoors),2029
13,Electronics,Electronics (Footwear),1127


## Step 4: Flags and derived fields (38 → 42)

In [15]:
s7 = cp.add_flags(s6.copy())
s8 = cp.add_derived_fields(s7.copy())
flags = ["is_late", "is_canceled", "is_suspected_fraud", "is_low_coverage_period"]
s8[flags].sum().to_frame("rows flagged").assign(share=lambda d: (d["rows flagged"] / len(s8)).map("{:.1%}".format))

,rows flagged,share
is_late,98977,54.8%
is_canceled,3692,2.0%
is_suspected_fraud,4062,2.3%
is_low_coverage_period,8557,4.7%


Flagged rows are **kept**. Each KPI chooses its filter explicitly (see `Model/handoff_plan.md`).

In [16]:
s8.groupby("shipping_mode")["shipping_delay_days"].agg(
    mean="mean", min="min", max="max", share_late=lambda d: (d > 0).mean()
).round(2).rename_axis("R15 shipping_delay_days")

,mean,min,max,share_late
R15 shipping_delay_days,,,,
First Class,1.00,1,1,1.00
Same Day,0.48,0,1,0.48
Second Class,1.99,0,4,0.80
Standard Class,-0.00,-2,2,0.40


Early signal for Question 02 (SLA realism): **First Class is late on every single item** (always 1 day over), while Standard Class meets its SLA on average.

## Result

In [17]:
clean = cp.clean(raw)
assert clean.equals(s8), "step-by-step result differs from cp.clean()"
print(f"raw   {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"clean {clean.shape[0]:,} rows x {clean.shape[1]} columns")

raw   180,519 rows x 53 columns
clean 180,519 rows x 42 columns


In [18]:
pd.DataFrame({
    "raw": [raw["Sales"].sum(), raw["Order Item Total"].sum(), raw["Order Profit Per Order"].sum(), raw["Order Id"].nunique(), len(raw)],
    "clean": [clean["sales"].sum(), clean["net_sales"].sum(), clean["profit"].sum(), clean["order_id"].nunique(), len(clean)],
}, index=["sales", "net_sales", "profit", "orders", "rows"]).round(2).assign(difference=lambda d: (d["clean"] - d["raw"]).round(2))

,raw,clean,difference
sales,36784735.01,36784734.31,-0.70
net_sales,33054402.38,33054402.04,-0.34
profit,3966902.97,3966902.97,0.00
orders,65752.00,65752.00,0.00
rows,180519.00,180519.00,0.00


Rows and orders are identical; money differs only by the float-noise rounding (R07). Cleaning changed **structure and labels, not the business numbers**.

In [19]:
clean.head()

,order_item_id,order_id,customer_id,product_id,category_id,department_id,order_date,shipping_date,order_status,payment_type,customer_segment,shipping_mode,days_shipping_real,days_shipping_scheduled,shipping_delay_days,delivery_status,is_late,product_name,product_price,category_name,department_name,quantity,sales,discount_amount,discount_rate,net_sales,profit,profit_ratio,market,order_region,order_country,order_state,order_city,customer_city,customer_state,customer_zipcode,customer_country,store_latitude,store_longitude,is_canceled,is_suspected_fraud,is_low_coverage_period
0,180517,77202,20755,1360,73,2,2018-01-31 22:56:00,2018-02-03 22:56:00,COMPLETE,DEBIT,Consumer,Standard Class,3,4,-1,Advance shipping,False,Smart watch,327.75,Sporting Goods,Fitness,1,327.75,13.11,0.04,314.64,91.25,0.29,Pacific Asia,Southeast Asia,Indonesia,Java Occidental,Bekasi,Caguas,PR,00725,Puerto Rico,18.251453,-66.037056,False,False,True
1,179254,75939,19492,1360,73,2,2018-01-13 12:27:00,2018-01-18 12:27:00,PENDING,TRANSFER,Consumer,Standard Class,5,4,1,Late delivery,True,Smart watch,327.75,Sporting Goods,Fitness,1,327.75,16.39,0.05,311.36,-249.09,-0.80,Pacific Asia,South Asia,India,Rajastán,Bikaner,Caguas,PR,00725,Puerto Rico,18.279451,-66.037064,False,False,True
2,179253,75938,19491,1360,73,2,2018-01-13 12:06:00,2018-01-17 12:06:00,CLOSED,CASH,Consumer,Standard Class,4,4,0,Shipping on time,False,Smart watch,327.75,Sporting Goods,Fitness,1,327.75,18.03,0.06,309.72,-247.78,-0.80,Pacific Asia,South Asia,India,Rajastán,Bikaner,San Jose,CA,95125,United States,37.292233,-121.881279,False,False,True
3,179252,75937,19490,1360,73,2,2018-01-13 11:45:00,2018-01-16 11:45:00,COMPLETE,DEBIT,Home Office,Standard Class,3,4,-1,Advance shipping,False,Smart watch,327.75,Sporting Goods,Fitness,1,327.75,22.94,0.07,304.81,22.86,0.08,Pacific Asia,Oceania,Australia,Queensland,Townsville,Los Angeles,CA,90027,United States,34.125946,-118.291016,False,False,True
4,179251,75936,19489,1360,73,2,2018-01-13 11:24:00,2018-01-15 11:24:00,PENDING_PAYMENT,PAYMENT,Corporate,Standard Class,2,4,-2,Advance shipping,False,Smart watch,327.75,Sporting Goods,Fitness,1,327.75,29.50,0.09,298.25,134.21,0.45,Pacific Asia,Oceania,Australia,Queensland,Townsville,Caguas,PR,00725,Puerto Rico,18.253769,-66.037048,False,False,True
